In [1]:
import pdfplumber
import pandas as pd

path = R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄 切割 共67頁.pdf"

# 定義表格偵測參數
table_settings = {
    "vertical_strategy": "lines",   # 強制改為根據「格線」切欄位，解決文字對不準問題
    "horizontal_strategy": "lines", # 根據橫線切行
    "explicit_vertical_lines": [],  # 如果有漏掉的線可以手動補，先留空
    "explicit_horizontal_lines": [],
    "snap_tolerance": 3,
    "join_tolerance": 3,
}

page_list = []

with pdfplumber.open(path) as pdf:
    for i, page in enumerate(pdf.pages):
        # 取得該頁所有表格
        tables = page.extract_tables(table_settings)
        
        for table in tables:
            df = pd.DataFrame(table)
            
            # --- 關鍵清洗開始 ---
            
            # 1. 移除全空行與全空欄
            df = df.dropna(how='all').dropna(axis=1, how='all')
            
            # 2. 處理儲存格內的換行符 (把 \n 換成空格)
            df = df.replace(r'\n', ' ', regex=True)
            
            # 3. 過濾掉太短的垃圾表格 (通常型錄一頁至少有 5 筆資料)
            if len(df) > 10:
                # 4. 清除「表頭垃圾」的條件過濾
                # 這裡建議用「包含某個關鍵字」來尋找真正的數據起始點
                # 找到包含 "剛性" 或 "油孔" 的最後一行 Index
                mask = df.apply(lambda row: row.astype(str).str.contains("剛性|Q|油孔").any(), axis=1)
                if mask.any():
                    header_idx = df[mask].index[-1]
                    df = df.iloc[header_idx + 1:].reset_index(drop=True).ffill()
                    a = df.iloc[:, :6]
                    b = df.iloc[:,-1]
                    df = pd.concat([a, b], axis = 1)
                                    
                page_list.append(df)
        
        print(f"第 {i+1} 頁處理完成")

# 合併所有 DataFrame
if page_list:
    # 注意：如果各頁欄位數量不同，concat 還是會參差不齊
    # 建議在 concat 前強制統一欄位數量
    # max_cols = max(len(d.columns) for d in page_list)
    # unified_list = [d.iloc[:, :max_cols] for d in page_list] # 截斷多餘欄位或補齊

    full_df = pd.concat(page_list, ignore_index=True)
    full_df.to_excel("PMI_Extracted_Standardized.xlsx", index=False)
    print("標準化表格擷取成功！")
else:
    print("沒抓到任何有效表格")

第 1 頁處理完成
第 2 頁處理完成
第 3 頁處理完成
第 4 頁處理完成
第 5 頁處理完成
第 6 頁處理完成
第 7 頁處理完成
第 8 頁處理完成
第 9 頁處理完成
第 10 頁處理完成
第 11 頁處理完成
第 12 頁處理完成
第 13 頁處理完成
第 14 頁處理完成
第 15 頁處理完成
第 16 頁處理完成
第 17 頁處理完成
第 18 頁處理完成
第 19 頁處理完成
第 20 頁處理完成
第 21 頁處理完成
第 22 頁處理完成
第 23 頁處理完成
第 24 頁處理完成
第 25 頁處理完成
第 26 頁處理完成
第 27 頁處理完成
第 28 頁處理完成
第 29 頁處理完成
第 30 頁處理完成
第 31 頁處理完成
第 32 頁處理完成
第 33 頁處理完成
第 34 頁處理完成
第 35 頁處理完成
第 36 頁處理完成
第 37 頁處理完成
第 38 頁處理完成
第 39 頁處理完成
第 40 頁處理完成
第 41 頁處理完成
第 42 頁處理完成
第 43 頁處理完成
第 44 頁處理完成
第 45 頁處理完成
第 46 頁處理完成
第 47 頁處理完成
第 48 頁處理完成
第 49 頁處理完成
第 50 頁處理完成
第 51 頁處理完成
第 52 頁處理完成
第 53 頁處理完成
第 54 頁處理完成
第 55 頁處理完成
第 56 頁處理完成
第 57 頁處理完成
第 58 頁處理完成
第 59 頁處理完成
第 60 頁處理完成
第 61 頁處理完成
第 62 頁處理完成
標準化表格擷取成功！


In [ ]:
import pandas as pd

# 1. 定義規則與欄位名稱
types = ["FSWC", "FDWC", "FSVC", "FDVC", "FOWC"]
groups = [5, 5, 4, 4, 2]
col_name = ["公稱 外徑", "導程", "珠徑", "珠卷數", "動負荷 C (kfg)", "靜負荷 Co (kfg)", "剛性 kfg/umk"]

# 假設 page_list 已經由之前的 pdfplumber 擷取出來
processed_dfs = []
current_idx = 0

# 2. 遍歷系列分組並注入標籤
for i, count in enumerate(groups):
    series_name = types[i]
    for _ in range(count):
        if current_idx < len(page_list):
            df_sub = page_list[current_idx].copy()
            
            # 確保欄位數量對齊
            df_sub = df_sub.iloc[:, :len(col_name)]
            df_sub.columns = col_name
            
            # 注入基礎標籤
            df_sub['brand'] = "PMI"
            df_sub['系列'] = series_name
            df_sub['category'] = "Screw"
            df_sub['data_type'] = "Specification" # 標註為規格資料
            
            processed_dfs.append(df_sub)
            current_idx += 1

# 3. 合併總表
if processed_dfs:
    full_df = pd.concat(processed_dfs, ignore_index=True)

    # 4. 格式化「型號」
    def format_model(row):
        try:
            dia = str(row["公稱 外徑"]).strip()
            lead = str(row["導程"]).strip()
            rigidity = str(row["剛性 kfg/umk"]).strip()
            if dia == "" or "nan" in dia.lower() or "None" in dia:
                return "N/A"
            return f"{dia}-{lead}-{rigidity}"
        except:
            return "N/A"

    full_df['型號'] = full_df.apply(format_model, axis=1)
    full_df = full_df[full_df['model_id'] != "N/A"].reset_index(drop=True)

    # 5. [新增] 產生語意欄位 (Semantic Text)
    def generate_semantic(row):
        return (
            f"這是銀泰 (PMI) 的滾珠螺桿規格。系列名稱為 {row['series']}，"
            f"完整型號為 {row['model_id']}。其主要參數如下：公稱外徑為 {row['公稱 外徑']} mm，"
            f"導程為 {row['導程']} mm，珠徑為 {row['珠徑']} mm，珠卷數為 {row['珠卷數']}。"
            f"在性能指標方面，其動負荷 (Ca) 為 {row['動負荷 C (kfg)']} kgf，"
            f"靜負荷 (Co) 為 {row['靜負荷 Co (kfg)']} kgf，剛性為 {row['剛性 kfg/umk']} kgf/umk。"
        )

    full_df['semantic_text'] = full_df.apply(generate_semantic, axis=1)

    # 6. 最後導出 Excel
    output_file = "PMI_Final_Data_V1.xlsx"
    full_df.to_excel(output_file, index=False)

    print(f"Success: Processed {len(processed_dfs)} tables.")
    print(f"Total rows with semantic text: {len(full_df)}.")
    print(f"File saved to {output_file}.")
else:
    print("Error: page_list is empty.")

Success: Processed 20 tables.
Total rows with semantic text: 590.
File saved to PMI_Final_Data_V1.xlsx.


In [1]:
### 文本擷取

import fitz  # PyMuPDF
import re


def step1_extract_text(pdf_path):
    """
    從 PDF 中提取文字，並進行初步的格式清理。
    """
    try:
        # 開啟 PDF 檔案
        doc = fitz.open(pdf_path)
        print(f"--- 檔案讀取成功：{pdf_path} ---")
        print(f"總頁數: {len(doc)}")
        
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # 提取文字
            raw_text = page.get_text("text")
            
            # 初步清理：移除多餘的連續空白、統一換行符
            clean_text = re.sub(r'\n\s*\n', '\n\n', raw_text) # 保持段落感
            clean_text = clean_text.strip()
            
            # 儲存結果（包含頁碼資訊，這對後續 RAG 引用非常重要）
            extracted_data.append({
                "page": page_num + 1,
                "content": clean_text
            })
            
            # 預覽前兩頁
            if page_num < 2:
                print(f"\n[第 {page_num + 1} 頁預覽]:")
                print(clean_text[:300] + "...") 
                print("-" * 30)

        doc.close()
        return extracted_data

    except Exception as e:
        print(f"讀取失敗：{e}")
        return None

# --- 執行處 ---
# 請將 'HIWIN_Catalog.pdf' 換成你實際的檔案路徑

raw_pages = step1_extract_text(R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄 切割 共67頁.pdf")

--- 檔案讀取成功：C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄 切割 共67頁.pdf ---
總頁數: 62

[第 1 頁預覽]:
BALLSCREWS
A129
規
格
FSIC
W
油孔Q
Z
X
H
60°
60°
W
油孔Q
60°
60°
W
油孔Q
60°
60°
G
Y
S
T
L
Ø
Ø
Ø
單位:mm
螺桿尺寸
鋼珠
直徑
循環
圈數
基本額定負荷(kgf)
螺帽
法蘭
配
合
螺絲孔
油孔
剛性
外徑導程
(1×10
6 REV.)
Ca(動負荷)Co(靜負荷) Dg6 
L
A
T
W
G
H
S
X
Y
Z
Q
kgf/
μm
14
3
2
3
260
460
26
37
46
10
36
-
-
10 4.5
8
4.5 M6×1P 13
4
2.381
3
420
805
26
42
46
10...
------------------------------

[第 2 頁預覽]:
A130
規
格
W
油孔Q
Z
X
H
60°
60°
W
油孔Q
60°
60°
W
油孔Q
60°
60°
G
Y
S
T
L
Ø
Ø
Ø
螺桿尺寸
鋼珠
直徑
循環
圈數
基本額定負荷(kgf)
螺帽
法蘭
配
合
螺絲孔
油孔
剛性
外徑導程
(1×10
6 REV.)
Ca(動負荷)Co(靜負荷) Dg6 
L
A
T
W
G
H
S
X
Y
Z
Q
kgf/
μm
32
4
2.381
3
560
1840
43
40
68
15
55
26
52
15 6.6
11
6.5 M8×1P 28
5
870
3070
49
45
5
3.175
3
1095
3060
48
47
...
------------------------------


In [ ]:
### 文本擷取，文字重新排列

import fitz
import pdfplumber
import pandas as pd
import re

#pdf_path = R"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf"
pdf_path = R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄 切割 共67頁.pdf"
doc_fitz = fitz.open(pdf_path)

with pdfplumber.open(pdf_path) as pdf_plumb:
        for i in range(len(pdf_plumb.pages)):
            page_fitz = doc_fitz[i]
            page = doc_fitz.load_page(i)
            
            # --- 關鍵修正：強制按「空間座標」排序文字 ---
            # get_text("words") 會回傳 (x0, y0, x1, y1, "word", ...)
            words = page_fitz.get_text("words")
            # 排序邏輯：優先比 y0 (高度)，y0 相近時(誤差3像素內)比 x0 (左右)
            words.sort(key=lambda w: (w[1] // 3, w[0]))
            
            # 重新組合排序後的文字列表
            sorted_text_list = [w[4] for w in words]
            full_sorted_text = "".join(sorted_text_list) # 拼成一個長字串

            if i < 20:
                print(f"\n[第 {i+1} 頁預覽]:")
                print(full_sorted_text[:300] + "...") 
                print("-" * 30)


[第 1 頁預覽]:
FSIC型號BALLSCREWSLTSZ60°60°60°60°60°60°YX油孔Q油孔Q油孔QH規G格WWW內循ØØØ環單位:mm系配列螺桿尺寸基本額定負荷(kgf)螺帽法蘭螺絲孔油孔剛性鋼珠循環合直徑圈數(1×106REV.)kgf/外徑導程Dg6LATWGHSXYZQCa(動負荷)Co(靜負荷)μm3232604602637461036--104.584.5M6×1P132.38134208054214144264610362040104.584.5M6×1P2.77848401870422153.1753720101026424610362040104.584.5M6×1P1642....
------------------------------

[第 2 頁預覽]:
FSIC型號BALLSCREWSLTSZ60°60°60°60°60°60°YX油孔Q油孔Q油孔Q規H格G內WWW循ØØØ環系單位:mm列配螺桿尺寸基本額定負荷(kgf)螺帽法蘭螺絲孔油孔剛性鋼珠循環合直徑圈數(1×106REV.)kgf/外徑導程Dg6LATWGHSXYZQCa(動負荷)Co(靜負荷)μm35601840402842.381436815552652156.6116.5M8×1P587030704945310953060473153.175414004080485373.512603060156.6116.5M8×1P416198061206260315003750533232...
------------------------------

[第 3 頁預覽]:
FSIC型號BALLSCREWSLTSZ60°60°60°60°60°60°YX油孔Q油孔Q油孔QH規格GWWW內循ØØØ環單位:mm系配列螺桿尺寸基本額定負荷(kgf)螺帽法蘭螺絲孔油孔剛性鋼珠循環合直徑圈數(1×106REV.)kgf/外徑導程Dg6LATWGHSXYZQCa(動負荷)Co(靜負荷)μm83.1754165060306172921675367215914.59M6×1P543416010750864845127.1447011016904284201117.511PT1/8"45330143309962166.35332208200701021101690428